# Boletins de urna — 2º turno das Eleições 2026

Análise exploratória dos dados abertos do TSE. Veja o `README.md` para a fonte,
a estrutura do projeto e as notas metodológicas.

**Para rodar:** ajuste `RAIZ` na célula abaixo e execute tudo. Os CSVs originais do
TSE não são necessários — a análise parte de `bu_2t.parquet`.

In [ ]:
import pandas as pd
import numpy as np
import glob, os

RAIZ = r"C:\Users\Lukan\Desktop\analise_urnas"   # <<< ajuste para a sua máquina
os.chdir(RAIZ)
print("pasta de trabalho:", os.getcwd())


## Regeneração do parquet *(opcional)*

Esta seção só roda se os CSVs do TSE estiverem em `raw/`. Eles não estão versionados
por causa do tamanho — 882 MB, com o de SP sozinho acima do limite de 100 MB por
arquivo do GitHub. Para baixá-los de novo, veja `raw/FONTE_DOS_DADOS.txt`.

Sem os CSVs, as células abaixo não fazem nada e a análise segue normalmente a partir
do parquet.

In [ ]:
arquivos = sorted(glob.glob(os.path.join(RAIZ, "raw", "bweb_2t_*.csv")))
print(len(arquivos), "CSVs encontrados em raw/")
if not arquivos:
    print("Nenhum CSV: as células desta seção serão puladas.")


In [ ]:
if arquivos:
    partes = []
    for f in arquivos:
        uf = os.path.basename(f).split("_")[2]
        df = pd.read_csv(f, sep=";", encoding="latin-1",dtype=str)
        df["UF_ARQUIVO"] = uf
        partes.append(df)
        print(uf, len(df))

    bu = pd.concat(partes, ignore_index=True)
    del partes
    print(bu.shape)


In [ ]:
if arquivos:
    bu.to_parquet(os.path.join(RAIZ, "bu_2t.parquet"), index=False)
    print("bu_2t.parquet regravado a partir dos CSVs")


In [ ]:
if arquivos:
    print(bu.columns.tolist())
    print(bu["UF_ARQUIVO"].value_counts().sort_index())
    pd.set_option("display.max_columns", None)
    bu.head()


In [ ]:
if arquivos:
    bu = bu.drop(columns=[
        "DT_GERACAO", "HH_GERACAO", "ANO_ELEICAO", "CD_TIPO_ELEICAO", "NM_TIPO_ELEICAO",
        "CD_PLEITO", "DT_PLEITO", "NR_TURNO", "CD_ELEICAO", "DS_ELEICAO", "CD_MUNICIPIO",
        "NR_PARTIDO", "NM_PARTIDO", "CD_TIPO_URNA", "CD_TIPO_VOTAVEL", "DS_TIPO_URNA",
        "CD_CARGA_1_URNA_EFETIVADA", "CD_CARGA_2_URNA_EFETIVADA",
        "CD_FLASHCARD_URNA_EFETIVADA", "DS_SECOES_AGREGADAS", "NR_JUNTA_APURADORA", "NR_TURMA_APURADORA",
    ])


In [ ]:
if arquivos:
    print(bu.shape)
    print(bu.columns.tolist())


## Ponto de entrada da análise

Daqui em diante tudo parte de `bu_2t.parquet`, que já traz as 2.088.107 linhas e as
46 colunas originais consolidadas das 28 UFs.

In [ ]:
bu = pd.read_parquet(os.path.join(RAIZ, "bu_2t.parquet"))
print(bu.shape)


In [8]:
import matplotlib.pyplot as plt

pres = bu[bu["DS_CARGO_PERGUNTA"].str.contains("Presidente", case=False, na=False)].copy()
pres["QT_VOTOS"] = pd.to_numeric(pres["QT_VOTOS"], errors="coerce")
pres["GRUPO"] = pres["NR_VOTAVEL"].map({"13": "Lula", "22": "Flavio", "95": "Brancos", "96": "Nulos"})
pres = pres.dropna(subset=["GRUPO"])

In [ ]:
PASTA = os.path.join(RAIZ, "plots")
os.makedirs(PASTA, exist_ok=True)

import numpy as np
import matplotlib.dates as mdates
from matplotlib.ticker import MultipleLocator, FuncFormatter, NullLocator

# Tema escuro: fundo preto na tela e no PNG salvo.
plt.style.use("dark_background")
plt.rcParams["grid.linestyle"] = ":"
plt.rcParams["figure.facecolor"] = "black"
plt.rcParams["axes.facecolor"] = "black"
plt.rcParams["savefig.facecolor"] = "black"
plt.rcParams["savefig.edgecolor"] = "black"

COR_LULA = "#e74c3c"
COR_FLAVIO = "#3498db"

def salvar(nome):
    plt.savefig(os.path.join(PASTA, f"{nome}.png"), dpi=150, bbox_inches="tight",
                facecolor="black", edgecolor="black")

def refinar(ax, x=None, y=None):
    ax.minorticks_on()
    for eixo, passos in ((ax.xaxis, x), (ax.yaxis, y)):
        if passos:
            eixo.set_major_locator(MultipleLocator(passos[0]))
            eixo.set_minor_locator(MultipleLocator(passos[1]))
    ax.grid(which="major", linestyle=":", linewidth=0.9, color="white", alpha=0.55)
    ax.grid(which="minor", linestyle=":", linewidth=0.5, color="white", alpha=0.25)
    ax.tick_params(which="minor", length=3)
    ax.set_axisbelow(True)


In [ ]:
tot = pres.groupby("GRUPO")["QT_VOTOS"].sum().reindex(["Lula", "Flavio", "Brancos", "Nulos"])
ax = tot.plot(kind="bar", figsize=(7, 4), color=[COR_LULA, COR_FLAVIO, "#bdc3c7", "#7f8c8d"])
ax.set_title("Total de votos por opção (Presidente, 2º turno)")
ax.set_ylabel("Votos")
plt.xticks(rotation=0)
refinar(ax)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1e6:.0f} mi"))
plt.tight_layout()
salvar("01_total_votos_por_opcao")
plt.show()

In [ ]:
uf = (pres[pres["GRUPO"].isin(["Lula", "Flavio"])]
      .pivot_table(index="UF_ARQUIVO", columns="GRUPO", values="QT_VOTOS", aggfunc="sum"))
pct = uf.div(uf.sum(axis=1), axis=0) * 100
pct = pct.sort_values("Lula")

pct[["Lula", "Flavio"]].plot(kind="barh", stacked=True, figsize=(8, 9), color=[COR_LULA, COR_FLAVIO])
plt.axvline(50, color="white", linestyle="--", linewidth=1)
plt.title("Percentual de votos válidos por UF")
plt.xlabel("%")
ax = plt.gca()
ax.set_xlim(0, 100)
refinar(ax, x=(5, 1))
ax.yaxis.set_minor_locator(NullLocator())
ax.grid(False, which="both", axis="y")
ax.legend(title="Candidato", loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
salvar("02_percentual_votos_validos_por_uf")
plt.show()

In [ ]:
secoes = pres.drop_duplicates(subset=["UF_ARQUIVO", "NM_MUNICIPIO", "NR_ZONA", "NR_SECAO", "NR_URNA_EFETIVADA"]).copy()
for c in ["QT_APTOS", "QT_ABSTENCOES"]:
    secoes[c] = pd.to_numeric(secoes[c], errors="coerce")

ab = secoes.groupby("UF_ARQUIVO")[["QT_ABSTENCOES", "QT_APTOS"]].sum()
taxa = (ab["QT_ABSTENCOES"] / ab["QT_APTOS"] * 100).sort_values()

taxa.plot(kind="barh", figsize=(8, 9), color="#95a5a6")
plt.title("Taxa de abstenção por UF")
plt.xlabel("%")
ax = plt.gca()
refinar(ax, x=(5, 1))
ax.yaxis.set_minor_locator(NullLocator())
ax.grid(False, which="both", axis="y")
plt.tight_layout()
salvar("03_taxa_abstencao_por_uf")
plt.show()

In [ ]:
recebido = pd.to_datetime(secoes["DT_BU_RECEBIDO"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
por_hora = recebido.dropna().dt.floor("h").value_counts().sort_index()

serie = por_hora.cumsum()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(serie.index, serie.values)
ax.set_title("Boletins recebidos (acumulado por hora)")
ax.set_ylabel("Boletins")
refinar(ax)
horas = (serie.index.max() - serie.index.min()) / pd.Timedelta(hours=1)
ax.xaxis.set_major_locator(mdates.HourLocator(interval=max(1, int(np.ceil(horas / 12)))))
ax.xaxis.set_minor_locator(mdates.MinuteLocator(byminute=range(0, 60, 10)) if horas <= 36 else mdates.HourLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M\n%d/%m"))
plt.tight_layout()
salvar("04_boletins_recebidos_por_hora")
plt.show()

In [ ]:
corte = pd.Timestamp("2026-10-25 17:00:00")

pres["RECEBIDO"] = pd.to_datetime(pres["DT_BU_RECEBIDO"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

apur = (pres[pres["GRUPO"].isin(["Lula", "Flavio"])]
        .dropna(subset=["RECEBIDO"])
        .assign(JANELA=lambda d: d["RECEBIDO"].dt.floor("10min"))
        .pivot_table(index="JANELA", columns="GRUPO", values="QT_VOTOS", aggfunc="sum", fill_value=0))

apur = apur.reindex(pd.date_range(apur.index.min(), apur.index.max(), freq="10min"), fill_value=0)
acum = apur.cumsum()


In [ ]:
# Prorrogação: votos em seções que encerraram a votação depois do fim nominal (17h de Brasília)
chave = ["UF_ARQUIVO", "NM_MUNICIPIO", "NR_ZONA", "NR_SECAO", "NR_URNA_EFETIVADA"]

votos = (pres[pres["GRUPO"].isin(["Lula", "Flavio"])]
         .pivot_table(index=chave, columns="GRUPO", values="QT_VOTOS", aggfunc="sum", fill_value=0)
         .reset_index())

sec = pres.drop_duplicates(subset=chave).copy()
sec["ENC"] = pd.to_datetime(sec["DT_ENCERRAMENTO"], errors="coerce")
sec["ABR"] = pd.to_datetime(sec["DT_ABERTURA"], errors="coerce")
sec["abr_hora"] = sec["ABR"].dt.hour

# Horários vêm no fuso local da UF: votação 8h-17h de Brasília = 6h-15h no AC, 7h-16h em AM/MS/MT/RO/RR.
# Hora nominal de abertura: padrão da UF, corrigido pela moda do município (AM e Noronha têm fuso próprio).
padrao = {"AC": 6, "AM": 7, "MS": 7, "MT": 7, "RO": 7, "RR": 7}
sec["abr_padrao"] = sec["UF_ARQUIVO"].map(padrao).fillna(8).astype(int)
mun = (sec.dropna(subset=["abr_hora"]).groupby(["UF_ARQUIVO", "NM_MUNICIPIO"])["abr_hora"]
       .agg(n="size", moda=lambda x: x.mode().iloc[0], share=lambda x: (x == x.mode().iloc[0]).mean())
       .reset_index())
mun["usa_moda"] = (mun["n"] >= 3) & (mun["share"] >= 0.5)
sec = sec.merge(mun[["UF_ARQUIVO", "NM_MUNICIPIO", "moda", "usa_moda"]], on=["UF_ARQUIVO", "NM_MUNICIPIO"], how="left")
sec["abr_nominal"] = np.where(sec["usa_moda"].eq(True), sec["moda"], sec["abr_padrao"]).astype(int)

sec["fim_nominal"] = sec["ENC"].dt.normalize() + pd.to_timedelta(sec["abr_nominal"] + 9, unit="h")
sec["atraso_min"] = (sec["ENC"] - sec["fim_nominal"]).dt.total_seconds() / 60
sec = sec.merge(votos, on=chave, how="left").fillna({"Lula": 0, "Flavio": 0}).dropna(subset=["atraso_min"])

faixas = [0, 1, 5, 10, 15, 30, 60, np.inf]
rotulos = ["0-1", "1-5", "5-10", "10-15", "15-30", "30-60", "60+"]
sec["FAIXA"] = pd.cut(sec["atraso_min"], bins=faixas, labels=rotulos, right=False)

por_faixa = sec.groupby("FAIXA", observed=True)[["Lula", "Flavio"]].sum()
por_faixa["secoes"] = sec.groupby("FAIXA", observed=True).size()
print(por_faixa)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 10), gridspec_kw={"height_ratios": [3, 2]})

# Painel de cima: proporção Lula x Flávio dentro de cada faixa de atraso (escala cheia 0-100%).
share = por_faixa[["Lula", "Flavio"]].div(por_faixa[["Lula", "Flavio"]].sum(axis=1), axis=0) * 100
geral = pres.groupby("GRUPO")["QT_VOTOS"].sum()
lula_geral = geral["Lula"] / (geral["Lula"] + geral["Flavio"]) * 100

x = np.arange(len(share))
ax1.bar(x, share["Lula"], 0.65, color=COR_LULA, label="Lula")
ax1.bar(x, share["Flavio"], 0.65, bottom=share["Lula"], color=COR_FLAVIO, label="Flavio")
ax1.axhline(50, color="white", linestyle="--", linewidth=1.2)
ax1.axhline(lula_geral, color="#f1c40f", linestyle="-.", linewidth=1.2,
            label=f"Lula no total geral ({lula_geral:.2f}%)")

for i, (f, linha) in enumerate(share.iterrows()):
    ax1.text(i, linha["Lula"] - 1.8, f"{linha['Lula']:.1f}%", ha="center", va="top",
             fontsize=9.5, color="white", fontweight="bold")
    ax1.text(i, 2, f"n={por_faixa.loc[f, 'secoes']:,}".replace(",", "."), ha="center", va="bottom",
             fontsize=7.5, color="white", alpha=0.8)

ax1.set_xticks(x)
ax1.set_xticklabels(share.index)
ax1.set_ylim(0, 100)
ax1.set_xlabel("Atraso no encerramento da seção, em minutos depois das 17h de Brasília")
ax1.set_ylabel("Proporção dos votos válidos na faixa (%)")
ax1.set_title("Quanto mais tarde a seção fecha, maior a fatia de Lula (prorrogação)")
ax1.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
refinar(ax1, y=(10, 2))
ax1.grid(False, which="both", axis="x")

atrasos = np.arange(0, 30.5, 0.5)
resto = pd.DataFrame({c: [sec.loc[sec["atraso_min"] > a, c].sum() / sec[c].sum() for a in atrasos]
                      for c in ["Lula", "Flavio"]}, index=atrasos)
ax2.plot(resto.index, resto["Lula"], color=COR_LULA, label="Lula")
ax2.plot(resto.index, resto["Flavio"], color=COR_FLAVIO, label="Flavio")
ax2.set_xlim(0, 30)
ax2.set_ylim(0, 1)
ax2.set_xlabel("Minutos depois das 17h de Brasília")
ax2.set_ylabel("Fração dos votos em seções\nencerradas depois desse atraso (0 a 1)")
ax2.legend()
refinar(ax2, x=(5, 1), y=(0.1, 0.05))

plt.tight_layout()
salvar("05_votos_prorrogacao_apos_17h")
plt.show()


In [ ]:
t = (acum.index - corte) / pd.Timedelta(hours=1)

fig = plt.figure(figsize=(11, 9))
ax = fig.add_subplot(111, projection="3d")
ax.plot(acum["Lula"], acum["Flavio"], t, color="gray", linewidth=0.8)
sc = ax.scatter(acum["Lula"], acum["Flavio"], t, c=t, cmap="viridis", s=10)

ax.set_xlabel("Votos Lula (acumulado)")
ax.set_ylabel("Votos Flávio (acumulado)")
ax.set_zlabel("Horas desde as 17:00")
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1e6:.0f} mi"))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1e6:.0f} mi"))
ax.set_zticks(np.arange(np.floor(t.min()), np.ceil(t.max()) + 0.5, 0.5))
for pane in (ax.xaxis, ax.yaxis, ax.zaxis):
    pane.set_pane_color((0, 0, 0, 1.0))
    pane._axinfo["grid"].update(color="white", linestyle=":", linewidth=0.6)
ax.view_init(elev=22, azim=-60)
fig.colorbar(sc, ax=ax, shrink=0.6, label="Horas desde as 17:00")

plt.tight_layout()
salvar("06_apuracao_3d_lula_flavio_tempo")
plt.show()

# Análise exploratória — gráficos 07 a 24

Seções temáticas conforme o roteiro do projeto. Cada gráfico tem sua interpretação logo abaixo, com as limitações dos dados.

In [ ]:
# ---- Tabelas-base para os gráficos 07 a 24 ----
# `sec`: uma linha por seção (chave UF+município+zona+seção), com comparecimento, horários e fusos.
# `vot`: uma linha por votável, para contagens de voto.
# Checado antes: a chave de 4 campos já é única (nenhuma seção tem duas urnas) e
# QT_APTOS = QT_COMPARECIMENTO + QT_ABSTENCOES em 100% das 499.205 seções.
import json
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
import matplotlib.patheffects as pe

CHAVE = ["SG_UF", "NM_MUNICIPIO", "NR_ZONA", "NR_SECAO"]

# Só as colunas usadas: `bu` inteiro tem 46 colunas de texto e copiá-lo
# duas vezes estoura a memória em máquinas com pouca RAM.
COLS_SEC = CHAVE + ["CD_MUNICIPIO", "NR_LOCAL_VOTACAO", "NR_URNA_EFETIVADA",
                    "QT_APTOS", "QT_COMPARECIMENTO", "QT_ABSTENCOES",
                    "QT_ELEI_BIOM_SEM_HABILITACAO", "DT_ABERTURA", "DT_ENCERRAMENTO",
                    "DT_EMISSAO_BU", "DT_BU_RECEBIDO"]
sec = (bu.loc[bu["DS_CARGO_PERGUNTA"] == "Presidente", COLS_SEC]
         .drop_duplicates(CHAVE).copy())
for c in ["QT_APTOS", "QT_COMPARECIMENTO", "QT_ABSTENCOES", "QT_ELEI_BIOM_SEM_HABILITACAO"]:
    sec[c] = pd.to_numeric(sec[c], errors="coerce")
for c in ["DT_ABERTURA", "DT_ENCERRAMENTO", "DT_EMISSAO_BU", "DT_BU_RECEBIDO"]:
    sec[c] = pd.to_datetime(sec[c], errors="coerce")
sec["TX_COMPARECIMENTO"] = sec["QT_COMPARECIMENTO"] / sec["QT_APTOS"] * 100
sec["TX_ABSTENCAO"] = sec["QT_ABSTENCOES"] / sec["QT_APTOS"] * 100

# Os horários vêm no fuso local da urna. A votação é das 8h às 17h de Brasília,
# o que é 7h-16h em AM/MS/MT/RO/RR e 6h-15h no AC. A hora nominal de abertura sai
# do padrão da UF, corrigido pela moda do município (o AM tem municípios em dois fusos).
padrao = {"AC": 6, "AM": 7, "MS": 7, "MT": 7, "RO": 7, "RR": 7}
sec["abr_padrao"] = sec["SG_UF"].map(padrao).fillna(8).astype(int)
_h = sec["DT_ABERTURA"].dt.hour
_mun = (sec.assign(h=_h).dropna(subset=["h"]).groupby(["SG_UF", "NM_MUNICIPIO"])["h"]
        .agg(n="size", moda=lambda x: x.mode().iloc[0], share=lambda x: (x == x.mode().iloc[0]).mean()).reset_index())
_mun["usa"] = (_mun["n"] >= 3) & (_mun["share"] >= 0.5)
sec = sec.merge(_mun[["SG_UF", "NM_MUNICIPIO", "moda", "usa"]], on=["SG_UF", "NM_MUNICIPIO"], how="left")
sec["ABR_NOMINAL"] = np.where(sec["usa"].eq(True), sec["moda"], sec["abr_padrao"]).astype(int)
sec["OFFSET_H"] = sec["ABR_NOMINAL"] - 8
for _c, _novo in [("DT_ABERTURA", "ABR_BSB"), ("DT_ENCERRAMENTO", "ENC_BSB"), ("DT_EMISSAO_BU", "EMI_BSB")]:
    sec[_novo] = sec[_c] - pd.to_timedelta(sec["OFFSET_H"], unit="h")
sec["DUR_H"] = (sec["DT_ENCERRAMENTO"] - sec["DT_ABERTURA"]).dt.total_seconds() / 3600
sec["ESPERA_BU_MIN"] = (sec["DT_EMISSAO_BU"] - sec["DT_ENCERRAMENTO"]).dt.total_seconds() / 60
sec["ATRASO_ENC_MIN"] = (sec["ENC_BSB"] - sec["ENC_BSB"].dt.normalize() - pd.Timedelta(hours=17)).dt.total_seconds() / 60
sec = sec.drop(columns=["abr_padrao", "moda", "usa"])

COLS_VOT = CHAVE + ["DS_CARGO_PERGUNTA", "DS_TIPO_VOTAVEL", "NR_VOTAVEL",
                    "NM_VOTAVEL", "SG_PARTIDO", "QT_VOTOS"]
vot = bu.loc[bu["DS_CARGO_PERGUNTA"] != "#NULO#", COLS_VOT].copy()
vot["QT_VOTOS"] = pd.to_numeric(vot["QT_VOTOS"], errors="coerce").fillna(0)

def gini(x):
    x = np.sort(np.asarray(x, dtype=float)); n = len(x)
    if n == 0 or x.sum() == 0: return np.nan
    return (2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum())

GEO = None
def mapa_ufs(ax, valores, cmap="viridis", titulo="", rotulo="", fmt="{:.1f}"):
    """Coroplético das UFs. `valores`: Series indexada pela sigla da UF."""
    global GEO
    if GEO is None:
        GEO = json.load(open(os.path.join(RAIZ, "dados", "br_ufs.geojson"), encoding="utf-8"))
    norm = plt.Normalize(valores.min(), valores.max()); cm = plt.get_cmap(cmap)
    patches, cores = [], []
    for feat in GEO["features"]:
        sig = feat["properties"]["sigla"]
        if sig not in valores.index:
            continue
        geom = feat["geometry"]
        polys = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
        maior, area = None, -1
        for poly in polys:
            anel = np.array(poly[0])
            patches.append(MplPolygon(anel, closed=True)); cores.append(cm(norm(valores[sig])))
            a = (anel[:, 0].max() - anel[:, 0].min()) * (anel[:, 1].max() - anel[:, 1].min())
            if a > area: area, maior = a, anel
        ax.annotate(f"{sig}\n{fmt.format(valores[sig])}", (maior[:, 0].mean(), maior[:, 1].mean()),
                    ha="center", va="center", fontsize=6.5, color="white", fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2, foreground="black")])
    ax.add_collection(PatchCollection(patches, facecolors=cores, edgecolors="white", linewidths=0.4))
    ax.autoscale_view(); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(titulo)
    sm = plt.cm.ScalarMappable(cmap=cm, norm=norm); sm.set_array([])
    plt.colorbar(sm, ax=ax, shrink=0.7, label=rotulo)

print("sec:", sec.shape, "| vot:", vot.shape)
print("memória — sec: %.0f MB | vot: %.0f MB"
      % (sec.memory_usage(deep=True).sum()/1e6, vot.memory_usage(deep=True).sum()/1e6))

# Daqui para baixo os gráficos usam só `sec` e `vot`. Se a memória apertar,
# libere o quadro grande:   del bu; import gc; gc.collect()


## 1. Participação democrática

Gráficos 07 a 10. Todas as taxas usam a tabela `sec`, com uma linha por seção, para não somar `QT_APTOS` repetido nas linhas de candidato.

In [ ]:
# 07. mapa de comparecimento por UF
uf = sec.groupby("SG_UF")[["QT_COMPARECIMENTO","QT_APTOS","QT_ABSTENCOES"]].sum()
uf["TX_COMP"] = uf["QT_COMPARECIMENTO"] / uf["QT_APTOS"] * 100
uf["TX_ABST"] = uf["QT_ABSTENCOES"] / uf["QT_APTOS"] * 100
nac = sec["QT_COMPARECIMENTO"].sum() / sec["QT_APTOS"].sum() * 100

fig, ax = plt.subplots(figsize=(10, 9))
mapa_ufs(ax, uf.loc[uf.index != "ZZ", "TX_COMP"], cmap="viridis",
         titulo=f"Taxa de comparecimento por UF (2º turno)\nmédia nacional {nac:.2f}%",
         rotulo="Comparecimento (%)", fmt="{:.1f}")
plt.tight_layout(); salvar("07_mapa_comparecimento_por_uf"); plt.close()


**07.** O comparecimento nacional foi de **80,78%**. O Ceará lidera com 86,1% e o Acre fecha a lista das UFs com 73,9%. O exterior (ZZ) é um caso à parte, com 41,6%. O padrão regional é claro: o Nordeste comparece mais que o Norte e o Centro-Oeste. O mapa descreve, não explica — a causa desse contraste não está nos dados do boletim.

In [ ]:
# 08. comparecimento x abstencao, 100% empilhado
o = uf.sort_values("TX_COMP")
fig, ax = plt.subplots(figsize=(9, 9))
ax.barh(o.index, o["TX_COMP"], color="#1abc9c", label="Comparecimento")
ax.barh(o.index, o["TX_ABST"], left=o["TX_COMP"], color="#e67e22", label="Abstenção")
ax.axvline(nac, color="white", linestyle="--", linewidth=1.2, label=f"Comparecimento nacional ({nac:.2f}%)")
for i, (sig, r) in enumerate(o.iterrows()):
    ax.text(r["TX_COMP"] - 1.5, i, f"{r['TX_COMP']:.1f}%", ha="right", va="center", fontsize=8, color="black", fontweight="bold")
ax.set_xlim(0, 100); ax.set_xlabel("% do eleitorado apto"); ax.set_ylabel("UF")
ax.set_title("Comparecimento x abstenção por UF (100% empilhado)")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
refinar(ax, x=(5, 1)); ax.yaxis.set_minor_locator(NullLocator()); ax.grid(False, which="both", axis="y")
plt.tight_layout(); salvar("08_comparecimento_x_abstencao_por_uf"); plt.close()


**08.** Em escala cheia de 0 a 100% as diferenças parecem pequenas, e é assim mesmo: entre a maior e a menor UF há 12 pontos percentuais. A barra do exterior é a única que destoa de verdade. Preferi não cortar o eixo para não exagerar a diferença visualmente.

In [ ]:
# 09. distribuicao do comparecimento por secao
tc = sec["TX_COMPARECIMENTO"].dropna()
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(tc, bins=np.arange(0, 100.5, 0.5), color="#1abc9c", alpha=0.75, label="Seções (bins de 0,5 pp)")
am = tc.sample(25000, random_state=0).to_numpy()
h = 1.06 * am.std() * len(am) ** (-1 / 5)          # regra de Silverman
xs = np.linspace(30, 100, 400)
dens = np.exp(-0.5 * ((xs[:, None] - am[None, :]) / h) ** 2).sum(axis=1) / (len(am) * h * np.sqrt(2 * np.pi))
ax.plot(xs, dens * len(tc) * 0.5, color="#f1c40f", linewidth=1.8, label=f"KDE gaussiana (h={h:.2f}, amostra de 25 mil)")
for q, cor in [(0.05, "#e74c3c"), (0.5, "white"), (0.95, "#e74c3c")]:
    ax.axvline(tc.quantile(q), color=cor, linestyle="--", linewidth=1.1)
    ax.annotate(f"p{int(q*100)}={tc.quantile(q):.1f}%", (tc.quantile(q), ax.get_ylim()[1]*0.92),
                rotation=90, fontsize=8, color=cor, ha="right", va="top")
ax.set_xlim(30, 100); ax.set_xlabel("Taxa de comparecimento da seção (%)"); ax.set_ylabel("Nº de seções")
ax.set_title(f"Distribuição do comparecimento por seção (n={len(tc):,})".replace(",", "."))
ax.legend(); refinar(ax, x=(5, 1))
plt.tight_layout(); salvar("09_distribuicao_comparecimento_por_secao"); plt.close()


**09.** A distribuição é aproximadamente simétrica em torno de 81,5% (mediana), com p05 em 71,2% e p95 em 89,0%. A cauda à esquerda é bem mais longa que a da direita: existem seções com comparecimento muito baixo, mas quase nenhuma com comparecimento perto de 100%. O KDE usa amostra de 25 mil seções com largura de banda pela regra de Silverman.

In [ ]:
# 10. eleitorado x comparecimento
m = sec[(sec["QT_APTOS"] > 0)].dropna(subset=["TX_COMPARECIMENTO"])
fig, ax = plt.subplots(figsize=(11, 6))
hb = ax.hexbin(m["QT_APTOS"], m["TX_COMPARECIMENTO"], gridsize=70, xscale="log", bins="log", cmap="inferno", mincnt=1)
bins = np.logspace(np.log10(m["QT_APTOS"].min()), np.log10(m["QT_APTOS"].max()), 25)
cat = pd.cut(m["QT_APTOS"], bins)
g = m.groupby(cat, observed=True)["TX_COMPARECIMENTO"].agg(["median", "size"])
g = g[g["size"] >= 100]            # só faixas com amostra suficiente
cen = [i.mid for i in g.index]
ax.plot(cen, g["median"].values, color="#1abc9c", linewidth=2.2, marker="o", markersize=4,
        label="Mediana por faixa (faixas com ≥100 seções)")
r = m["QT_APTOS"].rank().corr(m["TX_COMPARECIMENTO"].rank())   # Spearman = Pearson sobre ranks
ax.set_xscale("log"); ax.set_ylim(40, 100)
ax.set_xlabel("Eleitores aptos na seção (escala log)"); ax.set_ylabel("Taxa de comparecimento (%)")
n_fmt = f"{len(m):,}".replace(",", ".")
ax.set_title(f"Tamanho do eleitorado x comparecimento (n={n_fmt} seções)\n"
             f"Spearman ρ={r:.3f}: tamanho da seção não prevê comparecimento")
plt.colorbar(hb, ax=ax, label="Nº de seções (log)")
ax.legend(); refinar(ax, y=(10, 2))
plt.tight_layout(); salvar("10_eleitorado_x_comparecimento"); plt.close()


**10.** Spearman ρ = **−0,007**, ou seja, nenhuma relação monotônica entre tamanho da seção e comparecimento. Isso responde a uma das perguntas do roteiro: o tamanho do eleitorado da seção **não** prevê a participação. A nuvem densa entre 200 e 400 aptos é o tamanho padrão das seções. A mediana só é traçada em faixas com pelo menos 100 seções, porque nos extremos a amostra é pequena demais.

## 2. Comportamento eleitoral

Gráficos 11 a 14. **Limitação importante:** este é um conjunto de 2º turno, com 14 candidatos e 8 partidos. Os itens do roteiro sobre voto de legenda, concentração entre candidatos e heatmap por partido pressupõem dados de 1º turno; eles foram produzidos assim mesmo, com a limitação anotada em cada um.

In [ ]:
# 11. ranking de candidatos
nom = vot[vot["DS_TIPO_VOTAVEL"] == "Nominal"]
rank = (nom.groupby(["NM_VOTAVEL", "SG_PARTIDO", "DS_CARGO_PERGUNTA"])["QT_VOTOS"].sum()
           .reset_index().sort_values("QT_VOTOS"))
cor_cargo = {"Presidente": "#f1c40f", "Governador": "#9b59b6"}
fig, ax = plt.subplots(figsize=(11, 6.5))
rot = [f"{r.NM_VOTAVEL}  ({r.SG_PARTIDO})" for r in rank.itertuples()]
ax.barh(rot, rank["QT_VOTOS"], color=[cor_cargo[c] for c in rank["DS_CARGO_PERGUNTA"]])
for i, q in enumerate(rank["QT_VOTOS"]):
    ax.text(q * 1.03, i, f"{q/1e6:.2f} mi" if q >= 1e6 else f"{q/1e3:.0f} mil",
            va="center", fontsize=8, color="white")
ax.set_xscale("log"); ax.set_xlim(1e5, 2e8)
ax.set_xlabel("Votos nominais (escala log)")
ax.set_title("Candidatos mais votados no 2º turno\nsó há 14 candidatos: 2 a presidente e 12 a governador em 6 UFs")
ax.legend(handles=[plt.Rectangle((0,0),1,1,color=c,label=k) for k,c in cor_cargo.items()], loc="lower right")
refinar(ax); ax.grid(False, which="both", axis="y")
plt.tight_layout(); salvar("11_ranking_candidatos_mais_votados"); plt.close()


**11.** O 2º turno tem apenas 14 candidatos: 2 à presidência e 12 a governador em 6 UFs. A escala é logarítmica porque os dois presidenciáveis estão duas ordens de grandeza acima dos candidatos estaduais — em escala linear as barras de governador sumiriam.

In [ ]:
# 12. composicao do voto por cargo
comp = vot.pivot_table(index="DS_CARGO_PERGUNTA", columns="DS_TIPO_VOTAVEL", values="QT_VOTOS", aggfunc="sum", fill_value=0)
comp = comp.reindex(columns=["Nominal", "Branco", "Nulo"]).loc[["Presidente", "Governador"]]
cores = ["#2ecc71", "#bdc3c7", "#e74c3c"]

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax, cargo in zip(axes, comp.index):
    val = comp.loc[cargo]; tot = val.sum()
    cunhas, _ = ax.pie(val, labels=None, colors=cores, startangle=90, counterclock=False,
                       wedgeprops=dict(width=0.42, edgecolor="black", linewidth=1.2))
    for cunha, (nome, q) in zip(cunhas, val.items()):
        pct = q / tot * 100
        ang = np.deg2rad((cunha.theta1 + cunha.theta2) / 2)
        txt = f"{nome}\n{pct:.2f}%  ({q/1e6:.2f} mi)"
        if pct >= 10:                      # fatia grande: rótulo no miolo da rosca
            ax.text(0, 0, txt, ha="center", va="center", fontsize=11, color="white", fontweight="bold")
        else:                              # fatia pequena: fora, com linha de chamada
            lado = 1 if np.cos(ang) >= 0 else -1
            ax.annotate(txt, xy=(np.cos(ang), np.sin(ang)),
                        xytext=(1.45 * lado, 1.15 if nome == "Nulo" else 1.45),
                        ha="left" if lado > 0 else "right", va="center", fontsize=9, color="white",
                        arrowprops=dict(arrowstyle="-", color="white", lw=0.9,
                                        connectionstyle="angle,angleA=0,angleB=90,rad=5"))
    ax.set_xlim(-2.1, 2.1); ax.set_ylim(-1.35, 1.8)
    ax.set_title(f"{cargo}\ntotal {tot/1e6:.1f} mi de votos depositados", y=0.97)
fig.suptitle("Composição do voto por cargo (2º turno)\nnão há voto de legenda em 2º turno: o eleitor só escolhe candidato, branco ou nulo", y=1.0)
plt.tight_layout(); salvar("12_composicao_do_voto_por_cargo"); plt.close()


**12.** Não existe voto de legenda em 2º turno, então o roteiro pede uma categoria que não pode existir aqui. Brancos e nulos somam 3,24% na presidência e 6,75% nos governos estaduais. A diferença entre os cargos sugere menor engajamento com a disputa estadual, mas isso é leitura, não medição.

In [ ]:
# 13. concentracao de votos
def gini(x):
    x = np.sort(np.asarray(x, dtype=float)); n = len(x)
    if n == 0 or x.sum() == 0: return np.nan
    return (2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum())

fig, (axA, axB) = plt.subplots(1, 2, figsize=(14, 6))

# A: concentracao ENTRE CANDIDATOS, por disputa
disputas = []
pr = nom[nom["DS_CARGO_PERGUNTA"] == "Presidente"].groupby("NM_VOTAVEL")["QT_VOTOS"].sum()
disputas.append(("Presidente (BR)", pr))
for u in sorted(nom.loc[nom["DS_CARGO_PERGUNTA"] == "Governador", "SG_UF"].unique()):
    g = nom[(nom["DS_CARGO_PERGUNTA"] == "Governador") & (nom["SG_UF"] == u)].groupby("NM_VOTAVEL")["QT_VOTOS"].sum()
    disputas.append((f"Governador {u}", g))
nomes = [d[0] for d in disputas]
topo = [d[1].max() / d[1].sum() * 100 for d in disputas]
ginis = [gini(d[1]) for d in disputas]
ordem = np.argsort(topo)
axA.barh([nomes[i] for i in ordem], [topo[i] for i in ordem], color="#16a085")
for k, i in enumerate(ordem):
    axA.text(topo[i] - 1.5, k, f"{topo[i]:.1f}%  (Gini {ginis[i]:.3f}, {len(disputas[i][1])} cand.)",
             ha="right", va="center", fontsize=8, color="black", fontweight="bold")
axA.axvline(50, color="white", linestyle="--", linewidth=1.2)
axA.set_xlim(0, 100); axA.set_xlabel("Fatia do candidato mais votado da disputa (%)")
axA.set_title("A. Concentração entre candidatos, por disputa\ncada disputa tem 2 candidatos, então a concentração é limitada")
refinar(axA, x=(10, 2)); axA.grid(False, which="both", axis="y")

# B: Lorenz TERRITORIAL (entre municipios)
mun = vot.groupby(["SG_UF", "NM_MUNICIPIO", "NR_VOTAVEL"])["QT_VOTOS"].sum().unstack(fill_value=0)
series = {"Lula (13)": mun.get("13"), "Flávio (22)": mun.get("22"),
          "Brancos (95)": mun.get("95"), "Nulos (96)": mun.get("96")}
cores_s = {"Lula (13)": COR_LULA, "Flávio (22)": COR_FLAVIO, "Brancos (95)": "#bdc3c7", "Nulos (96)": "#f39c12"}
axB.plot([0, 1], [0, 1], color="white", linestyle="--", linewidth=1.2, label="Igualdade perfeita")
for k, ser in series.items():
    x = np.sort(ser.dropna().to_numpy(dtype=float))
    cum = np.concatenate([[0], np.cumsum(x) / x.sum()])
    axB.plot(np.linspace(0, 1, len(cum)), cum, color=cores_s[k], linewidth=1.9, label=f"{k} — Gini {gini(x):.3f}")
axB.set_xlim(0, 1); axB.set_ylim(0, 1)
axB.set_xlabel("Fração acumulada dos municípios (do menor para o maior)")
axB.set_ylabel("Fração acumulada dos votos")
axB.set_title(f"B. Concentração territorial dos votos\ncurva de Lorenz entre {len(mun):,} municípios".replace(",", "."))
axB.legend(loc="upper left", fontsize=8.5); refinar(axB, x=(0.1, 0.02), y=(0.1, 0.02))
plt.tight_layout(); salvar("13_concentracao_votos_lorenz"); plt.close()

for n_, t_, g_ in zip(nomes, topo, ginis): print(f"  {n_:<20} {t_:5.1f}%  Gini {g_:.3f}")


**13.** O painel A mostra o limite do 2º turno: com 2 candidatos por disputa, o Gini entre candidatos é quase nulo e a concentração fica presa entre 50% e 66%. O painel B é onde a curva de Lorenz tem resolução de verdade: entre os 5.715 municípios, os votos são altamente concentrados (Gini de 0,72 a 0,77), o que reflete a distribuição do próprio eleitorado brasileiro. Os votos de Lula são levemente menos concentrados (0,718) que os de Flávio (0,753).

In [ ]:
# 14. heatmap municipio x partido
nom = vot[vot["DS_TIPO_VOTAVEL"] == "Nominal"].copy()
nom["MUN"] = nom["NM_MUNICIPIO"] + "/" + nom["SG_UF"]

tot_mun = nom.groupby("MUN")["QT_VOTOS"].sum().sort_values(ascending=False)
top = tot_mun.head(45).index
piv = (nom[nom["MUN"].isin(top)].pivot_table(index="MUN", columns="SG_PARTIDO", values="QT_VOTOS", aggfunc="sum", fill_value=0))
pct = piv.div(piv.sum(axis=1), axis=0) * 100
pct = pct.reindex(tot_mun.loc[top].index)                       # ordem: maior eleitorado em cima
pct = pct[pct.sum().sort_values(ascending=False).index]          # colunas: partido mais votado à esquerda

fig, ax = plt.subplots(figsize=(10, 13))
im = ax.imshow(pct.values, cmap="magma", aspect="auto", vmin=0, vmax=100)
ax.set_xticks(range(len(pct.columns))); ax.set_xticklabels(pct.columns, rotation=45, ha="right")
ax.set_yticks(range(len(pct.index))); ax.set_yticklabels(pct.index, fontsize=7.5)
for i in range(pct.shape[0]):
    for j in range(pct.shape[1]):
        val = pct.values[i, j]
        if val > 0.05:
            ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=6.5,
                    color="white" if val < 55 else "black", fontweight="bold")
ax.set_xlabel("Partido"); ax.set_ylabel("Município (ordem: mais votos nominais em cima)")
ax.set_title("Percentual dos votos nominais por partido\n45 municípios com mais votos — 2º turno só tem PT e PL fora das 6 UFs com governador")
plt.colorbar(im, ax=ax, shrink=0.5, label="% dos votos nominais do município")
ax.set_xticks(np.arange(-.5, len(pct.columns), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(pct.index), 1), minor=True)
ax.grid(which="minor", color="black", linewidth=0.6); ax.tick_params(which="minor", length=0)
plt.tight_layout(); salvar("14_heatmap_municipio_x_partido"); plt.close()


**14.** Só 9 dos 45 municípios têm mais de dois partidos, e todos estão nas 6 UFs que também elegeram governador. Fora delas, o heatmap só pode mostrar PT e PL, porque é o que o 2º turno presidencial oferece. O item do roteiro pressupõe dados de 1º turno, com dezenas de partidos.

## 3. Desigualdade territorial e distribuição eleitoral

Gráficos 15 a 17.

In [ ]:
# 15. boxplot de comparecimento por UF
ordem = sec.groupby("SG_UF")["TX_COMPARECIMENTO"].median().sort_values().index
dados = [sec.loc[sec["SG_UF"] == u, "TX_COMPARECIMENTO"].dropna().to_numpy() for u in ordem]
fig, ax = plt.subplots(figsize=(13, 7))
bp = ax.boxplot(dados, tick_labels=ordem, showfliers=True, patch_artist=True, widths=0.65,
                flierprops=dict(marker=".", markersize=1.5, alpha=0.25, markerfacecolor="#7f8c8d", markeredgecolor="none"),
                medianprops=dict(color="#f1c40f", linewidth=2),
                whiskerprops=dict(color="white", linewidth=0.9), capprops=dict(color="white", linewidth=0.9))
cm = plt.get_cmap("viridis"); meds = sec.groupby("SG_UF")["TX_COMPARECIMENTO"].median()
norm = plt.Normalize(meds.min(), meds.max())
for caixa, u in zip(bp["boxes"], ordem):
    caixa.set(facecolor=cm(norm(meds[u])), alpha=0.85, edgecolor="white", linewidth=0.8)
nac = sec["QT_COMPARECIMENTO"].sum() / sec["QT_APTOS"].sum() * 100
ax.axhline(nac, color="white", linestyle="--", linewidth=1.2, label=f"Comparecimento nacional ({nac:.2f}%)")
ax.set_ylim(40, 102); ax.set_xlabel("UF (ordenada pela mediana)"); ax.set_ylabel("Taxa de comparecimento da seção (%)")
ax.set_title("Comparecimento por seção, distribuição dentro de cada UF\ncaixa = quartis, linha amarela = mediana, pontos = seções atípicas")
ax.legend(loc="lower right"); refinar(ax, y=(10, 2)); ax.grid(False, which="both", axis="x")
plt.tight_layout(); salvar("15_boxplot_comparecimento_por_uf"); plt.close()


**15.** As caixas mostram que a variação **dentro** de cada UF é maior que a variação **entre** UFs: quase toda UF tem seções de 65% a 95%. A ordem das medianas reproduz o mapa do gráfico 07. O exterior aparece com uma caixa enorme e deslocada, por ter poucas seções e realidades muito diferentes entre cidades.

In [ ]:
# 16. bolhas por municipio
mun = sec.groupby(["SG_UF", "NM_MUNICIPIO"]).agg(
    aptos=("QT_APTOS", "sum"), comp=("QT_COMPARECIMENTO", "sum"), abst=("QT_ABSTENCOES", "sum"), secoes=("NR_SECAO", "size")).reset_index()
mun["TX_ABST"] = mun["abst"] / mun["aptos"] * 100
mun["TX_COMP"] = mun["comp"] / mun["aptos"] * 100
fig, ax = plt.subplots(figsize=(12.5, 7))
sc = ax.scatter(mun["aptos"], mun["TX_ABST"], s=np.sqrt(mun["aptos"]) / 7, c=mun["TX_ABST"],
                cmap="plasma", alpha=0.55, edgecolors="none")
nac_ab = sec["QT_ABSTENCOES"].sum() / sec["QT_APTOS"].sum() * 100
ax.axhline(nac_ab, color="white", linestyle="--", linewidth=1.2, label=f"Abstenção nacional ({nac_ab:.2f}%)")
def rotular(linhas, cor, dys):
    for r, dy in zip(linhas.itertuples(), dys):
        ax.annotate(f"{r.NM_MUNICIPIO}/{r.SG_UF}", (r.aptos, r.TX_ABST), fontsize=8, color=cor,
                    xytext=(-34, dy), textcoords="offset points", ha="right", va="center",
                    arrowprops=dict(arrowstyle="-", color=cor, lw=0.7, alpha=0.8))
rotular(mun.nlargest(4, "aptos"), "white", [52, 20, -16, -48])
rotular(mun[mun["aptos"] > 20000].nlargest(3, "TX_ABST"), "#f1c40f", [30, 2, -26])
ax.set_xscale("log"); ax.set_xlabel("Eleitorado apto do município (escala log)"); ax.set_ylabel("Taxa de abstenção (%)")
ax.set_title(f"Municípios: eleitorado x abstenção (n={len(mun):,})\ntamanho e cor da bolha = eleitorado e abstenção".replace(",", "."))
plt.colorbar(sc, ax=ax, label="Taxa de abstenção (%)"); ax.legend(loc="upper right"); refinar(ax, y=(5, 1))
plt.tight_layout(); salvar("16_bolhas_municipio_eleitorado_abstencao"); plt.close()


**16.** A nuvem é horizontal: municípios de 2 mil e de 10 milhões de eleitores têm abstenção parecida, em torno de 19%. Os pontos amarelos no alto são todos do exterior (Nova York, Madri, Milão, Miami), com abstenção acima de 60%. As capitais estão próximas da média nacional.

In [ ]:
# 17. municipios com mais abstencoes absolutas
top = mun.nlargest(25, "abst").sort_values("abst")
rot = [f"{r.NM_MUNICIPIO}/{r.SG_UF}" for r in top.itertuples()]
fig, ax = plt.subplots(figsize=(11, 8))
b = ax.barh(rot, top["abst"] / 1e3, color=plt.get_cmap("plasma")(plt.Normalize(mun["TX_ABST"].min(), mun["TX_ABST"].max())(top["TX_ABST"])))
for i, r in enumerate(top.itertuples()):
    ax.text(r.abst / 1e3 * 1.01, i, f"{r.abst/1e3:,.0f} mil  ({r.TX_ABST:.1f}% do eleitorado)".replace(",", "."),
            va="center", fontsize=8, color="white")
ax.set_xlim(0, top["abst"].max() / 1e3 * 1.35)
ax.set_xlabel("Eleitores ausentes (milhares)"); ax.set_ylabel("Município")
ax.set_title("25 municípios com mais eleitores ausentes, em número absoluto\no ranking reflete sobretudo o tamanho do eleitorado, não a maior taxa de abstenção")
refinar(ax); ax.grid(False, which="both", axis="y")
plt.tight_layout(); salvar("17_municipios_mais_abstencoes_absolutas"); plt.close()

mun.to_pickle("municipios.pkl")


**17.** Este ranking mede número absoluto, não taxa, então ele basicamente ordena os municípios por tamanho: São Paulo lidera com 1,86 milhão de ausentes, mas sua taxa (20,4%) está perto da média nacional. Para encontrar onde a abstenção é proporcionalmente alta, o gráfico 16 é o indicado.

## 4. Funcionamento e transparência do sistema eleitoral

Gráficos 18 a 21. Os horários foram convertidos para o fuso de Brasília antes de qualquer comparação regional.

In [ ]:
# 18. duracao de funcionamento das urnas
d = sec["DUR_H"].dropna()
fig, ax = plt.subplots(figsize=(11.5, 5.5))
ax.hist(d, bins=np.arange(3.5, 15.5, 1/12), color="#3498db", alpha=0.8)
ax.set_yscale("log")
for q, cor, lab, dx, alt in [(0.01, "#e74c3c", "p01", -0.55, 0.30), (0.5, "#f1c40f", "mediana", 0.55, 0.030),
                             (0.99, "#e74c3c", "p99", 0.45, 0.30)]:
    x = d.quantile(q); ax.axvline(x, color=cor, linestyle="--", linewidth=1.2)
    ax.annotate(f"{lab} = {x:.2f} h", (x, ax.get_ylim()[1] * alt), xytext=(x + dx, ax.get_ylim()[1] * alt),
                fontsize=8.5, color=cor, ha="center", va="center",
                bbox=dict(boxstyle="round,pad=0.25", fc="black", ec=cor, lw=0.7),
                arrowprops=dict(arrowstyle="-", color=cor, lw=0.7))
ax.axvline(9, color="white", linestyle="-", linewidth=1.4, alpha=0.8)
ax.annotate("9 h previstas (8h às 17h)", (9, 1.5), xytext=(6.2, 60), fontsize=8.5, color="white",
            ha="center", va="center", bbox=dict(boxstyle="round,pad=0.3", fc="black", ec="white", lw=0.7),
            arrowprops=dict(arrowstyle="->", color="white", lw=0.8))
ax.set_xlim(3.5, 15.5); ax.set_xlabel("Duração entre abertura e encerramento da urna (horas)")
ax.set_ylabel("Nº de seções (escala log)")
ax.set_title(f"Tempo de funcionamento das urnas (n={len(d):,} seções)".replace(",", "."))
refinar(ax, x=(1, 0.25))
plt.tight_layout(); salvar("18_duracao_funcionamento_urnas"); plt.close()


**18.** A mediana é 9,03 h, exatamente as 9 horas previstas. A escala é logarítmica para que as caudas apareçam. Chamam atenção dois picos: um em 9h (o normal) e outro em 11h, que corresponde às seções com prorrogação de 2 horas já identificadas no gráfico 05. Durações abaixo de 8h são raríssimas.

In [ ]:
# 19. encerramento -> emissao do BU
e = sec["ESPERA_BU_MIN"].dropna()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [3, 2]})
a1.hist(e, bins=np.arange(0, 30.25, 0.25), color="#9b59b6", alpha=0.85)
for q, cor in [(0.5, "#f1c40f"), (0.95, "#e74c3c"), (0.99, "#e74c3c")]:
    x = e.quantile(q); a1.axvline(x, color=cor, linestyle="--", linewidth=1.2)
    a1.annotate(f"p{int(q*100)}={x:.1f} min", (x, a1.get_ylim()[1]*0.95), rotation=90, fontsize=8, color=cor, ha="right", va="top")
a1.set_xlim(0, 30); a1.set_xlabel("Minutos entre encerramento e emissão do boletim"); a1.set_ylabel("Nº de seções")
a1.set_title(f"Eficiência operacional: espera até emitir o BU\nmediana {e.median():.2f} min, máximo {e.max():.0f} min")
refinar(a1, x=(5, 1))
por_uf = sec.groupby("SG_UF")["ESPERA_BU_MIN"].median().sort_values()
a2.barh(por_uf.index, por_uf.values, color=plt.get_cmap("plasma")(plt.Normalize(por_uf.min(), por_uf.max())(por_uf.values)))
a2.axvline(e.median(), color="white", linestyle="--", linewidth=1.2, label=f"Mediana nacional ({e.median():.2f} min)")
for i, val in enumerate(por_uf.values):
    a2.text(val - 0.04, i, f"{val:.2f}", va="center", ha="right", fontsize=7, color="black", fontweight="bold")
a2.set_xlim(0, por_uf.max() * 1.12)
a2.set_xlabel("Mediana de minutos até emitir o BU"); a2.set_title("Por UF (eixo começa em zero)")
a2.legend(fontsize=8, loc="lower right"); refinar(a2, x=(0.5, 0.1)); a2.grid(False, which="both", axis="y")
a2.tick_params(axis="y", labelsize=8)
plt.tight_layout(); salvar("19_tempo_ate_emissao_do_bu"); plt.close()


**19.** A emissão do boletim é rápida e uniforme: mediana de 2,00 minutos, p99 de 5,2 minutos. A diferença entre UFs é pequena, de 1,37 a 2,13 minutos de mediana — o eixo começa em zero justamente para não inflar essa diferença. O máximo de 165 minutos é um caso isolado.

In [ ]:
# 20. biometria sem habilitacao x comparecimento
b = sec.dropna(subset=["QT_ELEI_BIOM_SEM_HABILITACAO", "QT_COMPARECIMENTO"])
b = b[b["QT_COMPARECIMENTO"] > 0].copy()
b["PCT_BIOM"] = b["QT_ELEI_BIOM_SEM_HABILITACAO"] / b["QT_COMPARECIMENTO"] * 100
zero = (b["QT_ELEI_BIOM_SEM_HABILITACAO"] == 0).mean()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5.8))
hb = a1.hexbin(b["QT_COMPARECIMENTO"], b["QT_ELEI_BIOM_SEM_HABILITACAO"], gridsize=60, bins="log", cmap="cividis", mincnt=1)
r = b["QT_COMPARECIMENTO"].rank().corr(b["QT_ELEI_BIOM_SEM_HABILITACAO"].rank())
a1.set_xlabel("Comparecimento na seção (eleitores)"); a1.set_ylabel("Eleitores sem habilitação biométrica")
a1.set_title(f"Biometria sem habilitação x comparecimento\nSpearman ρ={r:.3f} — {zero:.1%} das seções não tiveram nenhum caso")
plt.colorbar(hb, ax=a1, label="Nº de seções (log)"); refinar(a1)
pu = b.groupby("SG_UF").apply(lambda g: g["QT_ELEI_BIOM_SEM_HABILITACAO"].sum() / g["QT_COMPARECIMENTO"].sum() * 100, include_groups=False).sort_values()
a2.barh(pu.index, pu.values, color=plt.get_cmap("cividis")(plt.Normalize(pu.min(), pu.max())(pu.values)))
nac_b = b["QT_ELEI_BIOM_SEM_HABILITACAO"].sum() / b["QT_COMPARECIMENTO"].sum() * 100
a2.axvline(nac_b, color="white", linestyle="--", linewidth=1.2, label=f"Nacional ({nac_b:.2f}%)")
a2.set_xlabel("% do comparecimento sem habilitação biométrica"); a2.set_title("Por UF")
a2.legend(fontsize=8); refinar(a2); a2.grid(False, which="both", axis="y"); a2.tick_params(axis="y", labelsize=8)
plt.tight_layout(); salvar("20_biometria_sem_habilitacao"); plt.close()


**20.** Spearman ρ = 0,218: há relação positiva, mas fraca, e boa parte dela é mecânica, já que seções com mais eleitores tendem a ter mais casos de qualquer tipo. Na média nacional, 6,73% de quem compareceu não foi habilitado pela biometria. Apenas 0,4% das seções não tiveram nenhum caso.

In [ ]:
# 21. heatmap de horarios de encerramento por UF
h = sec.dropna(subset=["ENC_BSB"]).copy()
h["HORA"] = h["ENC_BSB"].dt.hour
tab = pd.crosstab(h["SG_UF"], h["HORA"])
tab = tab.reindex(columns=range(int(h["HORA"].min()), int(h["HORA"].max()) + 1), fill_value=0)
pct = tab.div(tab.sum(axis=1), axis=0) * 100
pct = pct.loc[sec.groupby("SG_UF")["ATRASO_ENC_MIN"].median().sort_values().index]
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(pct.values, cmap="inferno", aspect="auto", vmin=0, vmax=100)
ax.set_xticks(range(len(pct.columns))); ax.set_xticklabels([f"{c}h" for c in pct.columns])
ax.set_yticks(range(len(pct.index))); ax.set_yticklabels(pct.index, fontsize=8.5)
for i in range(pct.shape[0]):
    for j in range(pct.shape[1]):
        val = pct.values[i, j]
        if val >= 0.05:
            ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=6.5,
                    color="white" if val < 55 else "black", fontweight="bold")
ax.set_xlabel("Hora de encerramento da seção, convertida para o horário de Brasília")
ax.set_ylabel("UF (ordenada pelo atraso mediano)")
ax.set_title("Horário de encerramento das seções por UF\n% das seções da UF em cada hora; encerramento previsto às 17h")
plt.colorbar(im, ax=ax, shrink=0.6, label="% das seções da UF")
ax.set_xticks(np.arange(-.5, len(pct.columns), 1), minor=True); ax.set_yticks(np.arange(-.5, len(pct.index), 1), minor=True)
ax.grid(which="minor", color="black", linewidth=0.6); ax.tick_params(which="minor", length=0)
plt.tight_layout(); salvar("21_heatmap_horario_encerramento_por_uf"); plt.close()


**21.** Depois de converter tudo para o horário de Brasília, a grande maioria das seções de todas as UFs encerra dentro da hora das 17h. O Rio de Janeiro é o caso mais destoante, com 7,4% das seções encerrando às 19h — consistente com a prorrogação vista no gráfico 05. Sem a conversão de fuso, AM, MS, MT, RO, RR e AC apareceriam falsamente como adiantados.

## 5. Análises estatísticas e investigativas adicionais

Gráficos 22 a 24.

In [ ]:
# 22. concentracao e fragmentacao partidaria
nom = vot[vot["DS_TIPO_VOTAVEL"] == "Nominal"]
def frag(g):
    p = g.groupby("SG_PARTIDO")["QT_VOTOS"].sum(); p = p / p.sum()
    return pd.Series({"F": 1 / (p ** 2).sum(), "HHI": (p ** 2).sum(), "top1": p.max() * 100, "n_part": len(p)})
pf = nom.groupby("SG_UF").apply(frag, include_groups=False).sort_values("F")
nac = frag(nom)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14.5, 8.5), gridspec_kw={"width_ratios": [1, 1.25]})
cores = np.where(pf["n_part"] > 2, "#9b59b6", "#34495e")
a1.barh(pf.index, pf["F"], color=cores)
a1.axvline(nac["F"], color="white", linestyle="--", linewidth=1.2, label=f"Nacional (F={nac['F']:.2f})")
a1.axvline(2, color="#f1c40f", linestyle=":", linewidth=1.4, label="F=2: dois partidos de igual tamanho")
for i, (u, r) in enumerate(pf.iterrows()):
    a1.text(r["F"] + 0.02, i, f"{r['F']:.2f} ({int(r['n_part'])}p)", va="center", fontsize=7.5, color="white")
a1.set_xlim(1.0, pf["F"].max() * 1.18)
a1.set_xlabel("Número efetivo de partidos  F = 1 / Σpi²"); a1.set_ylabel("UF")
a1.set_title("Fragmentação partidária por UF\nroxo = UF que também elegeu governador; F<2 = um partido domina")
a1.legend(fontsize=8, loc="lower right"); refinar(a1, x=(0.2, 0.05)); a1.grid(False, which="both", axis="y")
a1.tick_params(axis="y", labelsize=8)
mapa_ufs(a2, pf.loc[pf.index != "ZZ", "F"], cmap="plasma",
         titulo=f"F por UF — nacional {nac['F']:.2f}\nem 2º turno o teto é baixo: só 8 partidos disputam",
         rotulo="Número efetivo de partidos", fmt="{:.2f}")
plt.tight_layout(); salvar("22_fragmentacao_partidaria"); plt.close()


**22.** F nacional é 2,21, ou seja, pouco mais de dois partidos efetivos — o esperado num 2º turno. UFs abaixo de 2 (como o Piauí, com 1,58) indicam domínio de um partido, não ausência de competição entre dois. As UFs com governador em disputa chegam a quase 4. Comparar este F com o de uma eleição de 1º turno seria enganoso.

In [ ]:
# 23. votos invalidos
inv = vot.pivot_table(index="SG_UF", columns="DS_TIPO_VOTAVEL", values="QT_VOTOS", aggfunc="sum", fill_value=0)
inv["total"] = inv.sum(axis=1)
inv["PCT_BRANCO"] = inv["Branco"] / inv["total"] * 100
inv["PCT_NULO"] = inv["Nulo"] / inv["total"] * 100
inv["PCT_INV"] = inv["PCT_BRANCO"] + inv["PCT_NULO"]
nac_inv = (vot.loc[vot["DS_TIPO_VOTAVEL"].isin(["Branco", "Nulo"]), "QT_VOTOS"].sum() / vot["QT_VOTOS"].sum() * 100)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 8.5), gridspec_kw={"width_ratios": [1, 1.25]})
o = inv.sort_values("PCT_INV")
a1.barh(o.index, o["PCT_BRANCO"], color="#bdc3c7", label="Brancos")
a1.barh(o.index, o["PCT_NULO"], left=o["PCT_BRANCO"], color="#e74c3c", label="Nulos")
a1.axvline(nac_inv, color="white", linestyle="--", linewidth=1.2, label=f"Nacional ({nac_inv:.2f}%)")
for i, (u, r) in enumerate(o.iterrows()):
    a1.text(r["PCT_INV"] + 0.15, i, f"{r['PCT_INV']:.1f}%", va="center", fontsize=7.5, color="white")
a1.set_xlabel("% do total de votos depositados"); a1.set_ylabel("UF")
a1.set_title("Votos brancos e nulos por UF")
a1.legend(fontsize=8, loc="lower right"); refinar(a1, x=(2, 0.5)); a1.grid(False, which="both", axis="y")
a1.tick_params(axis="y", labelsize=8)
mapa_ufs(a2, inv.loc[inv.index != "ZZ", "PCT_INV"], cmap="magma",
         titulo=f"Votos inválidos por UF — nacional {nac_inv:.2f}%", rotulo="Brancos + nulos (%)", fmt="{:.1f}")
plt.tight_layout(); salvar("23_votos_invalidos_por_uf"); plt.close()


**23.** Brancos e nulos somam 3,49% nacionalmente, variando de 1,76% em Roraima a 6,95% no Acre. Há padrão geográfico: a região Norte concentra os percentuais mais altos. Nulos superam brancos em praticamente todas as UFs.

In [ ]:
# 24. desigualdade de participacao entre secoes
def gini(x):
    x = np.sort(np.asarray(x, dtype=float)); n = len(x)
    return (2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum())
g = sec.dropna(subset=["TX_COMPARECIMENTO"]).groupby("SG_UF")["TX_COMPARECIMENTO"].agg(
    n="size", mediana="median", dp="std",
    p10=lambda x: x.quantile(.10), p90=lambda x: x.quantile(.90), gini=gini)
g["CV"] = g["dp"] / sec.groupby("SG_UF")["TX_COMPARECIMENTO"].mean() * 100
g["IQ80"] = g["p90"] - g["p10"]
g = g.sort_values("IQ80")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 8.5), gridspec_kw={"width_ratios": [1.1, 1.25]})
a1.hlines(g.index, g["p10"], g["p90"], color="#1abc9c", linewidth=5, alpha=0.75)
a1.plot(g["mediana"], g.index, "o", color="#f1c40f", markersize=5, label="Mediana da UF")
for i, (u, r) in enumerate(g.iterrows()):
    a1.text(r["p90"] + 0.7, i, f"{r['IQ80']:.1f} pp", va="center", fontsize=7.5, color="white")
a1.set_xlabel("Taxa de comparecimento da seção (%)"); a1.set_ylabel("UF (ordenada pela amplitude p10-p90)")
a1.set_title("Desigualdade de participação entre seções\nbarra = intervalo p10 a p90 das seções da UF")
a1.legend(fontsize=8, loc="lower right"); refinar(a1, x=(5, 1)); a1.grid(False, which="both", axis="y")
a1.tick_params(axis="y", labelsize=8)
mapa_ufs(a2, g.loc[g.index != "ZZ", "IQ80"], cmap="cividis",
         titulo="Amplitude p10-p90 do comparecimento entre seções\nquanto maior, mais desigual a participação dentro da UF",
         rotulo="Amplitude p10-p90 (pontos percentuais)", fmt="{:.1f}")
plt.tight_layout(); salvar("24_desigualdade_participacao_entre_secoes"); plt.close()


**24.** A amplitude p10-p90 mede quão desigual é a participação **entre seções** da mesma UF. O Distrito Federal é o mais homogêneo (8,2 pp) e o Amapá o mais desigual entre os estados (21,0 pp). O Norte aparece consistentemente mais desigual, o que combina com os municípios de difícil acesso dessa região. O exterior, com 32,2 pp, não é comparável ao resto.